# Tree based Models

Created a new Notebook only for Tree Based models to do Feature Engineering for Supplier and Product Identifier individual for Tree Based Methods: Trees dont care about the relationship of numbers, so it is possible to just give each supplier a random number (just the way it is now, but with smaller numbers to make computation faster), Neural Networks would think that there are relationships in Suppliers with similar numbers, therefore need other feature engineering

In [35]:
# ============================================================
# Setup: Install and import all required packages
# ============================================================

# Run this once if scikit-learn is not yet installed
# !pip install scikit-learn --break-system-packages

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import (
    classification_report,
    f1_score,
    cohen_kappa_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

In [36]:
# ============================================================
# Load ML-ready datasets
# ============================================================
import pandas as pd
import numpy as np

df_company_a = pd.read_csv('../data/formated/company_a_ML.csv')
df_company_b = pd.read_csv('../data/formated/company_b_ML.csv')

print("Company A shape:", df_company_a.shape)
print("Company B shape:", df_company_b.shape)

# ============================================================
# Label Encoding for Supplier and Unique Product Identifier
# Suitable for tree-based models (Decision Tree, Random Forest, etc.)
# Trees only use these values for splits, so arbitrary numeric labels
# do not introduce a false ordering problem like they would for NNs.
# ============================================================

def label_encode_column(df, column):
    codes = {value: i for i, value in enumerate(df[column].unique())}
    df[column] = df[column].map(codes)
    return df, codes

df_company_a, supplier_codes_a = label_encode_column(df_company_a, 'Supplier')
df_company_a, product_codes_a = label_encode_column(df_company_a, 'Unique Product Identifier')

df_company_b, supplier_codes_b = label_encode_column(df_company_b, 'Supplier')
df_company_b, product_codes_b = label_encode_column(df_company_b, 'Unique Product Identifier')

df_company_a.head()

Company A shape: (4459, 22)
Company B shape: (14896, 22)


,Unique Order Identifier,Order Complexity,Supplier,Unique Product Identifier,Planned Year,Planned Month,Planned Quarter,Planned Day,Planned Weekday,Arrival Year,...,Arrival Day,Arrival Weekday,Delivery Delay,Delivery Status,Ordered Quantity,Delivered Quantity,Delivery_Delay_Days,Status,Quantity Difference,Quantity Status
0,74682-11,17,0,0,2023,1,1,5,3,2022,...,15,3,-21,too early,3.0,3.0,-21,early,0.0,exact
1,74682-14,17,0,1,2023,1,1,5,3,2022,...,15,3,-21,too early,2.0,2.0,-21,early,0.0,exact
2,74666-2,7,1,2,2023,1,1,4,2,2023,...,4,2,0,on time,1.0,1.0,0,on_time,0.0,exact
3,74666-4,7,1,2,2023,1,1,4,2,2023,...,4,2,0,on time,1.0,1.0,0,on_time,0.0,exact
4,74806-1,1,2,3,2023,1,1,30,0,2023,...,4,2,-26,too early,4.0,4.0,-26,early,0.0,exact


## Quality Metrics

For each prediction, we check two simple things:

**1. Precision** – when the model predicts a class (e.g. "on time"), how often
is it actually correct?

**2. Overfitting check** – we compare how well the model does on data it was
trained on vs. data it has never seen. If it does much better on the training
data, it just memorized examples instead of learning real patterns – and it
won't work well on new orders in practice.

Example:
Train Recall: 0.98 | Test Recall: 0.64 | Gap: 0.34 → High overfitting risk

A small gap (< 0.05) is fine. A large gap (> 0.15) means the model is
overfitting and needs to be simplified (e.g. lower max_depth).

In [37]:
# ============================================================
# Evaluation function - overfitting check based on macro-averaged score
# so it doesn't get hidden by an imbalanced majority class
# ============================================================
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd

def evaluate_model(model, X_train, y_train, X_test, y_test, target_cols, model_name):
    print(f"=== {model_name} ===\n")

    summary = []  # collects one row per target for easy model comparison

    y_pred_train_all = model.predict(X_train)
    y_pred_test_all = model.predict(X_test)
    y_pred_train_df = pd.DataFrame(y_pred_train_all, columns=target_cols, index=y_train.index)
    y_pred_test_df = pd.DataFrame(y_pred_test_all, columns=target_cols, index=y_test.index)

    for col in target_cols:
        y_pred_train = y_pred_train_df[col]
        y_pred_test = y_pred_test_df[col]

        print(f"--- {col} ---")
        classes = sorted(y_test[col].unique())

        # All precision values in one block:
        # per-class + weighted (overall, frequency-weighted) + macro (every class equal)
        precisions = precision_score(y_test[col], y_pred_test, labels=classes, average=None, zero_division=0)
        test_precision_weighted = precision_score(y_test[col], y_pred_test, average='weighted', zero_division=0)
        test_precision_macro = precision_score(y_test[col], y_pred_test, average='macro', zero_division=0)

        print("Precision:")
        for cls, prec in zip(classes, precisions):
            print(f"  {cls}: {prec:.2f}")
        print(f"  weighted: {test_precision_weighted:.2f}")
        print(f"  macro:    {test_precision_macro:.2f}")

        test_f1_macro = f1_score(y_test[col], y_pred_test, average='macro', zero_division=0)

        # Macro recall treats every class equally, regardless of how common it is.
        # This catches cases where a model looks fine on paper (high accuracy)
        # but is actually memorizing rare classes instead of generalizing.
        train_recall = recall_score(y_train[col], y_pred_train, average='macro')
        test_recall = recall_score(y_test[col], y_pred_test, average='macro')
        gap = train_recall - test_recall

        print(f"\nF1 (macro): {test_f1_macro:.2f}")
        print(f"Train Recall (macro): {train_recall:.2f} | Test Recall (macro): {test_recall:.2f}")

        if gap < 0.05:
            verdict = "Low overfitting risk"
        elif gap < 0.15:
            verdict = "Moderate overfitting risk"
        else:
            verdict = "High overfitting risk"
        print(f"Overfitting check: {verdict} (gap: {gap:.2f})\n")

        summary.append({
            'Target': col,
            'Precision (weighted)': round(test_precision_weighted, 3),
            'Precision (macro)': round(test_precision_macro, 3),
            'Recall (macro)': round(test_recall, 3),
            'F1 (macro)': round(test_f1_macro, 3),
            'Overfit gap': round(gap, 3)
        })

    return pd.DataFrame(summary)

## Train/Test Split and Feature Selection

Target variables (y): Delivery Status, Quantity Status

Features (X): only information that is known **before** the delivery arrives.
Columns like Delivery Delay, Quantity Difference, Delivered Quantity, and all
Arrival Date features are excluded, since they are only known after the delivery
has already happened and would leak the answer directly into the model.

In [38]:
# ============================================================
# Train/Test Split - Company A and Company B
# ============================================================
from sklearn.model_selection import train_test_split

feature_cols = [
    'Order Complexity', 'Supplier', 'Unique Product Identifier',
    'Planned Year', 'Planned Month', 'Planned Quarter', 'Planned Day', 'Planned Weekday',
    'Ordered Quantity'
]
target_cols = ['Delivery Status', 'Quantity Status']

# Company A
X_a = df_company_a[feature_cols]
y_a = df_company_a[target_cols]
X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(
    X_a, y_a, test_size=0.2, random_state=42
)

# Company B
X_b = df_company_b[feature_cols]
y_b = df_company_b[target_cols]
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_b, y_b, test_size=0.2, random_state=42
)

print("Company A - Train:", X_train_a.shape, "Test:", X_test_a.shape)
print("Company B - Train:", X_train_b.shape, "Test:", X_test_b.shape)

Company A - Train: (3567, 9) Test: (892, 9)
Company B - Train: (11916, 9) Test: (2980, 9)


## Grid Search

In [39]:
# ============================================================
# Grid Search setup - shared scorer for multi-output models
# Uses macro recall (averaged across both target columns) so the
# grid search optimizes for the same thing evaluate_model checks
# ============================================================
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, recall_score

def macro_recall_multioutput(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    scores = [
        recall_score(y_true[:, i], y_pred[:, i], average='macro')
        for i in range(y_true.shape[1])
    ]
    return np.mean(scores)

multioutput_scorer = make_scorer(macro_recall_multioutput)

## Decision Tree

A Decision Tree is a simple supervised learning model that makes predictions
by splitting the data into branches based on feature values. Each split is a
decision rule, and the final leaf gives the predicted class.

Main parameters:
- **max_depth**: limits how deep the tree grows (controls overfitting)
- **min_samples_split / min_samples_leaf**: minimum samples required to split/form a leaf
- **criterion**: function used to measure split quality (Gini or entropy)

Possible issues: Decision Trees can easily overfit and are sensitive to small
changes in the data.

In [40]:
# ============================================================
# Decision Tree - Company A (Grid Search)
# ============================================================
from sklearn.tree import DecisionTreeClassifier
from sklearn.multioutput import MultiOutputClassifier

param_grid_dt = {
    'estimator__max_depth': [15, 20, 25, 30, None],
    'estimator__min_samples_split': [2, 5, 10],
    'estimator__min_samples_leaf': [1, 2, 4],
    'estimator__criterion': ['gini', 'entropy']
}

dt_grid_a = GridSearchCV(
    MultiOutputClassifier(DecisionTreeClassifier(random_state=42)),
    param_grid=param_grid_dt,
    scoring=multioutput_scorer,
    cv=5,
    n_jobs=-1
)
dt_grid_a.fit(X_train_a, y_train_a)

print("Best hyperparameters (Decision Tree - Company A):", dt_grid_a.best_params_)
dt_model_a = dt_grid_a.best_estimator_

evaluate_model(dt_model_a, X_train_a, y_train_a, X_test_a, y_test_a, target_cols, "Decision Tree - Company A")

Best hyperparameters (Decision Tree - Company A): {'estimator__criterion': 'gini', 'estimator__max_depth': 25, 'estimator__min_samples_leaf': 1, 'estimator__min_samples_split': 2}
=== Decision Tree - Company A ===

--- Delivery Status ---
Precision:
  on time: 0.83
  too early: 0.85
  too late: 0.92
  weighted: 0.89
  macro:    0.87

F1 (macro): 0.87
Train Recall (macro): 0.99 | Test Recall (macro): 0.88
Overfitting check: Moderate overfitting risk (gap: 0.11)

--- Quantity Status ---
Precision:
  exact: 0.99
  over delivered: 1.00
  under delivered: 0.17
  weighted: 0.98
  macro:    0.72

F1 (macro): 0.68
Train Recall (macro): 0.93 | Test Recall (macro): 0.65
Overfitting check: High overfitting risk (gap: 0.27)



,Target,Precision (weighted),Precision (macro),Recall (macro),F1 (macro),Overfit gap
0,Delivery Status,0.891,0.869,0.878,0.873,0.112
1,Quantity Status,0.981,0.718,0.654,0.683,0.272


In [41]:
# ============================================================
# Decision Tree - Company B (Grid Search)
# ============================================================
dt_grid_b = GridSearchCV(
    MultiOutputClassifier(DecisionTreeClassifier(random_state=42)),
    param_grid=param_grid_dt,
    scoring=multioutput_scorer,
    cv=5,
    n_jobs=-1
)
dt_grid_b.fit(X_train_b, y_train_b)

print("Best hyperparameters (Decision Tree - Company B):", dt_grid_b.best_params_)
dt_model_b = dt_grid_b.best_estimator_

evaluate_model(dt_model_b, X_train_b, y_train_b, X_test_b, y_test_b, target_cols, "Decision Tree - Company B")

Best hyperparameters (Decision Tree - Company B): {'estimator__criterion': 'gini', 'estimator__max_depth': None, 'estimator__min_samples_leaf': 1, 'estimator__min_samples_split': 2}
=== Decision Tree - Company B ===

--- Delivery Status ---
Precision:
  on time: 0.56
  too early: 0.68
  too late: 0.78
  weighted: 0.71
  macro:    0.67

F1 (macro): 0.68
Train Recall (macro): 1.00 | Test Recall (macro): 0.68
Overfitting check: High overfitting risk (gap: 0.32)

--- Quantity Status ---
Precision:
  exact: 0.96
  over delivered: 0.76
  under delivered: 0.27
  weighted: 0.93
  macro:    0.67

F1 (macro): 0.65
Train Recall (macro): 0.99 | Test Recall (macro): 0.63
Overfitting check: High overfitting risk (gap: 0.36)



,Target,Precision (weighted),Precision (macro),Recall (macro),F1 (macro),Overfit gap
0,Delivery Status,0.713,0.674,0.678,0.676,0.321
1,Quantity Status,0.935,0.666,0.634,0.649,0.360


## Random Forest

A Random Forest is an ensemble model that combines many Decision Trees. Each
tree is trained on a random subset of the data, and the final prediction is
made by majority vote across all trees. This usually makes it more accurate
and more stable than a single Decision Tree.

Main parameters:
- **n_estimators**: number of trees in the forest
- **max_depth**: maximum depth of each tree
- **class_weight='balanced'**: gives more weight to rare classes, which is
  important here since both targets are imbalanced

In [42]:
# ============================================================
# Random Forest - Company A (Grid Search)
# ============================================================
from sklearn.ensemble import RandomForestClassifier

param_grid_rf = {
    'estimator__n_estimators': [100, 200, 300],
    'estimator__max_depth': [10, 15, 20, 25, None],
    'estimator__min_samples_split': [2, 5, 10],
    'estimator__min_samples_leaf': [1, 2, 4],
    'estimator__class_weight': ['balanced']
}

rf_grid_a = GridSearchCV(
    MultiOutputClassifier(RandomForestClassifier(random_state=42, n_jobs=-1)),
    param_grid=param_grid_rf,
    scoring=multioutput_scorer,
    cv=5,
    n_jobs=-1
)
rf_grid_a.fit(X_train_a, y_train_a)

print("Best hyperparameters (Random Forest - Company A):", rf_grid_a.best_params_)
rf_model_a = rf_grid_a.best_estimator_

evaluate_model(rf_model_a, X_train_a, y_train_a, X_test_a, y_test_a, target_cols, "Random Forest - Company A")

Best hyperparameters (Random Forest - Company A): {'estimator__class_weight': 'balanced', 'estimator__max_depth': 25, 'estimator__min_samples_leaf': 1, 'estimator__min_samples_split': 5, 'estimator__n_estimators': 200}
=== Random Forest - Company A ===

--- Delivery Status ---
Precision:
  on time: 0.92
  too early: 0.82
  too late: 0.95
  weighted: 0.91
  macro:    0.90

F1 (macro): 0.90
Train Recall (macro): 0.98 | Test Recall (macro): 0.91
Overfitting check: Moderate overfitting risk (gap: 0.07)

--- Quantity Status ---
Precision:
  exact: 0.99
  over delivered: 1.00
  under delivered: 0.25
  weighted: 0.98
  macro:    0.75

F1 (macro): 0.70
Train Recall (macro): 0.99 | Test Recall (macro): 0.67
Overfitting check: High overfitting risk (gap: 0.32)



,Target,Precision (weighted),Precision (macro),Recall (macro),F1 (macro),Overfit gap
0,Delivery Status,0.911,0.899,0.913,0.905,0.068
1,Quantity Status,0.983,0.747,0.672,0.701,0.322


In [43]:
# ============================================================
# Random Forest - Company B (Grid Search)
# ============================================================
rf_grid_b = GridSearchCV(
    MultiOutputClassifier(RandomForestClassifier(random_state=42, n_jobs=-1)),
    param_grid=param_grid_rf,
    scoring=multioutput_scorer,
    cv=5,
    n_jobs=-1
)
rf_grid_b.fit(X_train_b, y_train_b)

print("Best hyperparameters (Random Forest - Company B):", rf_grid_b.best_params_)
rf_model_b = rf_grid_b.best_estimator_

evaluate_model(rf_model_b, X_train_b, y_train_b, X_test_b, y_test_b, target_cols, "Random Forest - Company B")

/opt/miniconda3/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best hyperparameters (Random Forest - Company B): {'estimator__class_weight': 'balanced', 'estimator__max_depth': 20, 'estimator__min_samples_leaf': 2, 'estimator__min_samples_split': 10, 'estimator__n_estimators': 300}
=== Random Forest - Company B ===

--- Delivery Status ---
Precision:
  on time: 0.56
  too early: 0.71
  too late: 0.81
  weighted: 0.74
  macro:    0.69

F1 (macro): 0.70
Train Recall (macro): 0.93 | Test Recall (macro): 0.71
Overfitting check: High overfitting risk (gap: 0.22)

--- Quantity Status ---
Precision:
  exact: 0.98
  over delivered: 0.62
  under delivered: 0.34
  weighted: 0.95
  macro:    0.64

F1 (macro): 0.70
Train Recall (macro): 0.98 | Test Recall (macro): 0.78
Overfitting check: High overfitting risk (gap: 0.20)



,Target,Precision (weighted),Precision (macro),Recall (macro),F1 (macro),Overfit gap
0,Delivery Status,0.741,0.695,0.708,0.701,0.225
1,Quantity Status,0.946,0.645,0.783,0.699,0.199


## XGBoost

XGBoost (Extreme Gradient Boosting) is an ensemble model that builds trees
sequentially, where each new tree corrects the errors of the previous ones.
It's generally faster and more accurate than a plain Random Forest, and
handles imbalanced and complex tabular data well.

Main parameters:
- **n_estimators**: number of boosting rounds (trees)
- **max_depth**: maximum depth of each tree
- **learning_rate**: how much each tree corrects the previous ones (lower = more conservative)
- **subsample**: fraction of training rows used per tree (helps prevent overfitting)

Possible issues: XGBoost can overfit with too many estimators/too high a
learning rate, and is more sensitive to hyperparameter tuning than Random Forest.

In [44]:
# ============================================================
# Setup: Install and import XGBoost, LightGBM, CatBoost
# ============================================================
import subprocess
import sys

def install_and_import_boosting_libraries():
    packages = {
        'xgboost': 'xgboost',
        'lightgbm': 'lightgbm',
        'catboost': 'catboost'
    }

    for import_name, pip_name in packages.items():
        try:
            __import__(import_name)
            print(f"{pip_name} already installed.")
        except ImportError:
            print(f"Installing {pip_name} ...")
            subprocess.check_call([
                sys.executable, "-m", "pip", "install", pip_name, "--break-system-packages"
            ])
            print(f"{pip_name} installed.")

install_and_import_boosting_libraries()

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

print("XGBoost, LightGBM, and CatBoost are ready to use.")

xgboost already installed.
lightgbm already installed.
catboost already installed.
XGBoost, LightGBM, and CatBoost are ready to use.


In [45]:
# ============================================================
# XGBoost needs integer-encoded target labels (0, 1, 2, ...),
# not strings. This helper encodes the targets, and the wrapper
# class decodes predictions back to the original strings so
# evaluate_model() still works exactly as before.
# ============================================================
from sklearn.preprocessing import LabelEncoder

def encode_targets(y_train, y_test, target_cols):
    encoders = {}
    y_train_enc = y_train.copy()
    y_test_enc = y_test.copy()
    for col in target_cols:
        le = LabelEncoder()
        y_train_enc[col] = le.fit_transform(y_train[col])
        y_test_enc[col] = le.transform(y_test[col])
        encoders[col] = le
    return y_train_enc, y_test_enc, encoders

class DecodedMultiOutputClassifier:
    # Wraps a fitted MultiOutputClassifier trained on label-encoded targets
    # and decodes predictions back to the original string labels.
    def __init__(self, fitted_model, encoders, target_cols):
        self.fitted_model = fitted_model
        self.encoders = encoders
        self.target_cols = target_cols

    def predict(self, X):
        y_pred_enc = self.fitted_model.predict(X)
        y_pred_enc = pd.DataFrame(y_pred_enc, columns=self.target_cols)
        y_pred_decoded = y_pred_enc.copy()
        for col in self.target_cols:
            y_pred_decoded[col] = self.encoders[col].inverse_transform(y_pred_enc[col])
        return y_pred_decoded.values

In [46]:
# ============================================================
# XGBoost - Company A (Grid Search)
# ============================================================
from xgboost import XGBClassifier

y_train_a_enc, y_test_a_enc, target_encoders_a = encode_targets(y_train_a, y_test_a, target_cols)

param_grid_xgb = {
    'estimator__n_estimators': [100, 300],
    'estimator__max_depth': [3, 10],
    'estimator__learning_rate': [0.01, 0.2],
    'estimator__subsample': [0.8, 1.0]
}

xgb_grid_a = GridSearchCV(
    MultiOutputClassifier(XGBClassifier(random_state=42, eval_metric='mlogloss')),
    param_grid=param_grid_xgb,
    scoring=multioutput_scorer,
    cv=5,
    n_jobs=-1
)
xgb_grid_a.fit(X_train_a, y_train_a_enc)

print("Best hyperparameters (XGBoost - Company A):", xgb_grid_a.best_params_)
xgb_model_a = DecodedMultiOutputClassifier(xgb_grid_a.best_estimator_, target_encoders_a, target_cols)

evaluate_model(xgb_model_a, X_train_a, y_train_a, X_test_a, y_test_a, target_cols, "XGBoost - Company A")

/opt/miniconda3/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best hyperparameters (XGBoost - Company A): {'estimator__learning_rate': 0.2, 'estimator__max_depth': 10, 'estimator__n_estimators': 300, 'estimator__subsample': 1.0}
=== XGBoost - Company A ===

--- Delivery Status ---
Precision:
  on time: 0.95
  too early: 0.85
  too late: 0.93
  weighted: 0.91
  macro:    0.91

F1 (macro): 0.91
Train Recall (macro): 0.99 | Test Recall (macro): 0.90
Overfitting check: Moderate overfitting risk (gap: 0.08)

--- Quantity Status ---
Precision:
  exact: 0.99
  over delivered: 1.00
  under delivered: 1.00
  weighted: 0.99
  macro:    1.00

F1 (macro): 0.70
Train Recall (macro): 0.94 | Test Recall (macro): 0.64
Overfitting check: High overfitting risk (gap: 0.30)



,Target,Precision (weighted),Precision (macro),Recall (macro),F1 (macro),Overfit gap
0,Delivery Status,0.913,0.913,0.903,0.908,0.082
1,Quantity Status,0.988,0.996,0.638,0.699,0.303


In [47]:
# ============================================================
# XGBoost - Company B (Grid Search)
# ============================================================
y_train_b_enc, y_test_b_enc, target_encoders_b = encode_targets(y_train_b, y_test_b, target_cols)

xgb_grid_b = GridSearchCV(
    MultiOutputClassifier(XGBClassifier(random_state=42, eval_metric='mlogloss', n_jobs=1)),
    param_grid=param_grid_xgb,
    scoring=multioutput_scorer,
    cv=5,
    n_jobs=4
)

xgb_grid_b.fit(X_train_b, y_train_b_enc)

print("Best hyperparameters (XGBoost - Company B):", xgb_grid_b.best_params_)
xgb_model_b = DecodedMultiOutputClassifier(xgb_grid_b.best_estimator_, target_encoders_b, target_cols)

evaluate_model(xgb_model_b, X_train_b, y_train_b, X_test_b, y_test_b, target_cols, "XGBoost - Company B")

Best hyperparameters (XGBoost - Company B): {'estimator__learning_rate': 0.2, 'estimator__max_depth': 10, 'estimator__n_estimators': 300, 'estimator__subsample': 1.0}
=== XGBoost - Company B ===

--- Delivery Status ---
Precision:
  on time: 0.74
  too early: 0.74
  too late: 0.80
  weighted: 0.77
  macro:    0.76

F1 (macro): 0.73
Train Recall (macro): 0.99 | Test Recall (macro): 0.72
Overfitting check: High overfitting risk (gap: 0.28)

--- Quantity Status ---
Precision:
  exact: 0.97
  over delivered: 0.85
  under delivered: 0.61
  weighted: 0.95
  macro:    0.81

F1 (macro): 0.72
Train Recall (macro): 0.97 | Test Recall (macro): 0.68
Overfitting check: High overfitting risk (gap: 0.29)



,Target,Precision (weighted),Precision (macro),Recall (macro),F1 (macro),Overfit gap
0,Delivery Status,0.770,0.761,0.716,0.733,0.276
1,Quantity Status,0.951,0.808,0.677,0.717,0.290


## LightGBM

LightGBM is another gradient boosting framework, similar in spirit to
XGBoost, but built to be faster on large datasets. It grows trees
leaf-wise (choosing the leaf that reduces loss the most) rather than
level-wise, which can make it more efficient but also more prone to
overfitting on smaller datasets if not tuned carefully.

Main parameters:
- **n_estimators**: number of boosting rounds (trees)
- **max_depth**: maximum depth of each tree (-1 means no limit)
- **learning_rate**: shrinkage applied to each tree's contribution
- **num_leaves**: maximum number of leaves per tree (main control for model complexity)

Possible issues: since LightGBM grows leaf-wise, a high num_leaves combined
with no max_depth limit can overfit quickly on smaller datasets.

In [48]:
# ============================================================
# LightGBM - Company A (Grid Search)
# ============================================================
from lightgbm import LGBMClassifier

param_grid_lgbm = {
    'estimator__n_estimators': [100, 300],
    'estimator__max_depth': [-1, 5, 15],
    'estimator__learning_rate': [0.01, 0.2],
    'estimator__num_leaves': [31, 70]
}

lgbm_grid_a = GridSearchCV(
    MultiOutputClassifier(LGBMClassifier(random_state=42, verbose=-1)),
    param_grid=param_grid_lgbm,
    scoring=multioutput_scorer,
    cv=5,
    n_jobs=-1
)
lgbm_grid_a.fit(X_train_a, y_train_a)

print("Best hyperparameters (LightGBM - Company A):", lgbm_grid_a.best_params_)
lgbm_model_a = lgbm_grid_a.best_estimator_

evaluate_model(lgbm_model_a, X_train_a, y_train_a, X_test_a, y_test_a, target_cols, "LightGBM - Company A")

Best hyperparameters (LightGBM - Company A): {'estimator__learning_rate': 0.2, 'estimator__max_depth': -1, 'estimator__n_estimators': 300, 'estimator__num_leaves': 31}
=== LightGBM - Company A ===

--- Delivery Status ---
Precision:
  on time: 0.96
  too early: 0.86
  too late: 0.93
  weighted: 0.91
  macro:    0.92

F1 (macro): 0.91
Train Recall (macro): 0.99 | Test Recall (macro): 0.90
Overfitting check: Moderate overfitting risk (gap: 0.08)

--- Quantity Status ---
Precision:
  exact: 0.99
  over delivered: 1.00
  under delivered: 1.00
  weighted: 0.99
  macro:    1.00

F1 (macro): 0.71
Train Recall (macro): 0.93 | Test Recall (macro): 0.66
Overfitting check: High overfitting risk (gap: 0.27)



,Target,Precision (weighted),Precision (macro),Recall (macro),F1 (macro),Overfit gap
0,Delivery Status,0.913,0.917,0.901,0.909,0.085
1,Quantity Status,0.989,0.996,0.656,0.710,0.270


In [49]:
# ============================================================
# LightGBM - Company B (Grid Search)
# ============================================================
lgbm_grid_b = GridSearchCV(
    MultiOutputClassifier(LGBMClassifier(random_state=42, verbose=-1)),
    param_grid=param_grid_lgbm,
    scoring=multioutput_scorer,
    cv=5,
    n_jobs=-1
)
lgbm_grid_b.fit(X_train_b, y_train_b)

print("Best hyperparameters (LightGBM - Company B):", lgbm_grid_b.best_params_)
lgbm_model_b = lgbm_grid_b.best_estimator_

evaluate_model(lgbm_model_b, X_train_b, y_train_b, X_test_b, y_test_b, target_cols, "LightGBM - Company B")

Best hyperparameters (LightGBM - Company B): {'estimator__learning_rate': 0.2, 'estimator__max_depth': 15, 'estimator__n_estimators': 300, 'estimator__num_leaves': 70}
=== LightGBM - Company B ===

--- Delivery Status ---
Precision:
  on time: 0.74
  too early: 0.75
  too late: 0.79
  weighted: 0.77
  macro:    0.76

F1 (macro): 0.73
Train Recall (macro): 0.99 | Test Recall (macro): 0.71
Overfitting check: High overfitting risk (gap: 0.28)

--- Quantity Status ---
Precision:
  exact: 0.96
  over delivered: 0.83
  under delivered: 0.64
  weighted: 0.95
  macro:    0.81

F1 (macro): 0.70
Train Recall (macro): 0.97 | Test Recall (macro): 0.64
Overfitting check: High overfitting risk (gap: 0.33)



,Target,Precision (weighted),Precision (macro),Recall (macro),F1 (macro),Overfit gap
0,Delivery Status,0.768,0.759,0.713,0.731,0.280
1,Quantity Status,0.950,0.810,0.642,0.696,0.331


## CatBoost

CatBoost is a gradient boosting library developed with native support for
categorical features and strong out-of-the-box performance with less tuning
than XGBoost or LightGBM. Since our categorical columns (Supplier, Unique
Product Identifier) have already been label-encoded for tree models, it's
used here as a straightforward boosted-tree alternative.

Main parameters:
- **iterations**: number of boosting rounds (trees) — CatBoost's equivalent of n_estimators
- **depth**: depth of each tree
- **learning_rate**: shrinkage applied to each tree's contribution

Possible issues: CatBoost training is generally slower per iteration than
XGBoost/LightGBM, so grid search here can take noticeably longer.

In [50]:
# ============================================================
# CatBoost Wrapper: gibt flache 1D-Predictions zurück,
# damit MultiOutputClassifier korrekt stapelt
# ============================================================
from catboost import CatBoostClassifier
import numpy as np

class CatBoostClassifierFlat(CatBoostClassifier):
    def predict(self, X, **kwargs):
        return np.asarray(super().predict(X, **kwargs)).ravel()

In [51]:
# ============================================================
# CatBoost - Company A (Grid Search)
# ============================================================
param_grid_catboost = {
    'estimator__iterations': [100, 300],
    'estimator__depth': [4, 10],
    'estimator__learning_rate': [0.01, 0.2]
}

catboost_grid_a = GridSearchCV(
    MultiOutputClassifier(CatBoostClassifierFlat(random_state=42, verbose=0)),
    param_grid=param_grid_catboost,
    scoring=multioutput_scorer,
    cv=5,
    n_jobs=-1
)

catboost_grid_a.fit(X_train_a, y_train_a)

print("Best hyperparameters (CatBoost - Company A):", catboost_grid_a.best_params_)
catboost_model_a = catboost_grid_a.best_estimator_

evaluate_model(catboost_model_a, X_train_a, y_train_a, X_test_a, y_test_a, target_cols, "CatBoost - Company A")

Best hyperparameters (CatBoost - Company A): {'estimator__depth': 10, 'estimator__iterations': 300, 'estimator__learning_rate': 0.2}
=== CatBoost - Company A ===

--- Delivery Status ---
Precision:
  on time: 0.91
  too early: 0.83
  too late: 0.91
  weighted: 0.89
  macro:    0.88

F1 (macro): 0.88
Train Recall (macro): 0.98 | Test Recall (macro): 0.87
Overfitting check: Moderate overfitting risk (gap: 0.11)

--- Quantity Status ---
Precision:
  exact: 0.99
  over delivered: 1.00
  under delivered: 0.50
  weighted: 0.99
  macro:    0.83

F1 (macro): 0.71
Train Recall (macro): 0.93 | Test Recall (macro): 0.67
Overfitting check: High overfitting risk (gap: 0.25)



,Target,Precision (weighted),Precision (macro),Recall (macro),F1 (macro),Overfit gap
0,Delivery Status,0.887,0.883,0.870,0.876,0.113
1,Quantity Status,0.986,0.830,0.673,0.713,0.253


In [52]:
# ============================================================
# CatBoost - Company B (Grid Search)
# ============================================================
catboost_grid_b = GridSearchCV(
    MultiOutputClassifier(CatBoostClassifierFlat(random_state=42, verbose=0, thread_count=1)),
    param_grid=param_grid_catboost,
    scoring=multioutput_scorer,
    cv=5,
    n_jobs=4
)
catboost_grid_b.fit(X_train_b, y_train_b)

print("Best hyperparameters (CatBoost - Company B):", catboost_grid_b.best_params_)
catboost_model_b = catboost_grid_b.best_estimator_

evaluate_model(catboost_model_b, X_train_b, y_train_b, X_test_b, y_test_b, target_cols, "CatBoost - Company B")

Best hyperparameters (CatBoost - Company B): {'estimator__depth': 10, 'estimator__iterations': 300, 'estimator__learning_rate': 0.2}
=== CatBoost - Company B ===

--- Delivery Status ---
Precision:
  on time: 0.70
  too early: 0.73
  too late: 0.76
  weighted: 0.74
  macro:    0.73

F1 (macro): 0.69
Train Recall (macro): 0.93 | Test Recall (macro): 0.67
Overfitting check: High overfitting risk (gap: 0.26)

--- Quantity Status ---
Precision:
  exact: 0.96
  over delivered: 0.75
  under delivered: 0.65
  weighted: 0.95
  macro:    0.79

F1 (macro): 0.64
Train Recall (macro): 0.87 | Test Recall (macro): 0.60
Overfitting check: High overfitting risk (gap: 0.27)



,Target,Precision (weighted),Precision (macro),Recall (macro),F1 (macro),Overfit gap
0,Delivery Status,0.738,0.729,0.668,0.689,0.259
1,Quantity Status,0.945,0.788,0.604,0.638,0.265


# Results WIP

In [53]:
# ============================================================
# Final model comparison - Company A
# ============================================================
results_a = pd.concat([
    evaluate_model(dt_model_a, X_train_a, y_train_a, X_test_a, y_test_a, target_cols, "Decision Tree - A").assign(Model="Decision Tree"),
    evaluate_model(rf_model_a, X_train_a, y_train_a, X_test_a, y_test_a, target_cols, "Random Forest - A").assign(Model="Random Forest"),
    evaluate_model(xgb_model_a, X_train_a, y_train_a, X_test_a, y_test_a, target_cols, "XGBoost - A").assign(Model="XGBoost"),
    evaluate_model(catboost_model_a, X_train_a, y_train_a, X_test_a, y_test_a, target_cols, "CatBoost - A").assign(Model="CatBoost"),
    # evaluate_model(lgbm_model_a, ..., "LightGBM - A").assign(Model="LightGBM"),  # dein 5. Modell
], ignore_index=True)

=== Decision Tree - A ===

--- Delivery Status ---
Precision:
  on time: 0.83
  too early: 0.85
  too late: 0.92
  weighted: 0.89
  macro:    0.87

F1 (macro): 0.87
Train Recall (macro): 0.99 | Test Recall (macro): 0.88
Overfitting check: Moderate overfitting risk (gap: 0.11)

--- Quantity Status ---
Precision:
  exact: 0.99
  over delivered: 1.00
  under delivered: 0.17
  weighted: 0.98
  macro:    0.72

F1 (macro): 0.68
Train Recall (macro): 0.93 | Test Recall (macro): 0.65
Overfitting check: High overfitting risk (gap: 0.27)

=== Random Forest - A ===

--- Delivery Status ---
Precision:
  on time: 0.92
  too early: 0.82
  too late: 0.95
  weighted: 0.91
  macro:    0.90

F1 (macro): 0.90
Train Recall (macro): 0.98 | Test Recall (macro): 0.91
Overfitting check: Moderate overfitting risk (gap: 0.07)

--- Quantity Status ---
Precision:
  exact: 0.99
  over delivered: 1.00
  under delivered: 0.25
  weighted: 0.98
  macro:    0.75

F1 (macro): 0.70
Train Recall (macro): 0.99 | Test Recal

In [54]:
# ============================================================
# Comparison tables - Company A
# ============================================================
print("=== F1 (macro) - Company A ===")
print(results_a.pivot(index='Model', columns='Target', values='F1 (macro)').round(3).to_string())

print("\n=== Precision (weighted) - Company A ===")
print(results_a.pivot(index='Model', columns='Target', values='Precision (weighted)').round(3).to_string())

print("\n=== Overfit gap - Company A ===")
print(results_a.pivot(index='Model', columns='Target', values='Overfit gap').round(3).to_string())

=== F1 (macro) - Company A ===
Target         Delivery Status  Quantity Status
Model                                          
CatBoost                 0.876            0.713
Decision Tree            0.873            0.683
Random Forest            0.905            0.701
XGBoost                  0.908            0.699

=== Precision (weighted) - Company A ===
Target         Delivery Status  Quantity Status
Model                                          
CatBoost                 0.887            0.986
Decision Tree            0.891            0.981
Random Forest            0.911            0.983
XGBoost                  0.913            0.988

=== Overfit gap - Company A ===
Target         Delivery Status  Quantity Status
Model                                          
CatBoost                 0.113            0.253
Decision Tree            0.112            0.272
Random Forest            0.068            0.322
XGBoost                  0.082            0.303


In [56]:
# ============================================================
# Final model comparison - Company B
# ============================================================
results_b = pd.concat([
    evaluate_model(dt_model_b, X_train_b, y_train_b, X_test_b, y_test_b, target_cols, "Decision Tree - B").assign(Model="Decision Tree"),
    evaluate_model(rf_model_b, X_train_b, y_train_b, X_test_b, y_test_b, target_cols, "Random Forest - B").assign(Model="Random Forest"),
    evaluate_model(xgb_model_b, X_train_b, y_train_b, X_test_b, y_test_b, target_cols, "XGBoost - B").assign(Model="XGBoost"),
    evaluate_model(catboost_model_b, X_train_b, y_train_b, X_test_b, y_test_b, target_cols, "CatBoost - B").assign(Model="CatBoost"),
    # evaluate_model(lgbm_model_b, ..., "LightGBM - B").assign(Model="LightGBM"),  # dein 5. Modell
], ignore_index=True)

=== Decision Tree - B ===

--- Delivery Status ---
Precision:
  on time: 0.56
  too early: 0.68
  too late: 0.78
  weighted: 0.71
  macro:    0.67

F1 (macro): 0.68
Train Recall (macro): 1.00 | Test Recall (macro): 0.68
Overfitting check: High overfitting risk (gap: 0.32)

--- Quantity Status ---
Precision:
  exact: 0.96
  over delivered: 0.76
  under delivered: 0.27
  weighted: 0.93
  macro:    0.67

F1 (macro): 0.65
Train Recall (macro): 0.99 | Test Recall (macro): 0.63
Overfitting check: High overfitting risk (gap: 0.36)

=== Random Forest - B ===

--- Delivery Status ---
Precision:
  on time: 0.56
  too early: 0.71
  too late: 0.81
  weighted: 0.74
  macro:    0.69

F1 (macro): 0.70
Train Recall (macro): 0.93 | Test Recall (macro): 0.71
Overfitting check: High overfitting risk (gap: 0.22)

--- Quantity Status ---
Precision:
  exact: 0.98
  over delivered: 0.62
  under delivered: 0.34
  weighted: 0.95
  macro:    0.64

F1 (macro): 0.70
Train Recall (macro): 0.98 | Test Recall (macro

In [58]:
# ============================================================
# Comparison tables - Company B
# ============================================================
print("=== F1 (macro) - Company B ===")
print(results_b.pivot(index='Model', columns='Target', values='F1 (macro)').round(3).to_string())

print("\n=== Precision (weighted) - Company B ===")
print(results_b.pivot(index='Model', columns='Target', values='Precision (weighted)').round(3).to_string())

print("\n=== Overfit gap - Company B ===")
print(results_b.pivot(index='Model', columns='Target', values='Overfit gap').round(3).to_string())

=== F1 (macro) - Company B ===
Target         Delivery Status  Quantity Status
Model                                          
CatBoost                 0.689            0.638
Decision Tree            0.676            0.649
Random Forest            0.701            0.699
XGBoost                  0.733            0.717

=== Precision (weighted) - Company B ===
Target         Delivery Status  Quantity Status
Model                                          
CatBoost                 0.738            0.945
Decision Tree            0.713            0.935
Random Forest            0.741            0.946
XGBoost                  0.770            0.951

=== Overfit gap - Company B ===
Target         Delivery Status  Quantity Status
Model                                          
CatBoost                 0.259            0.265
Decision Tree            0.321            0.360
Random Forest            0.225            0.199
XGBoost                  0.276            0.290
